In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from neo4j import GraphDatabase
import torch.optim as Adam

# 1. Database Connection
uri = "bolt://localhost:7687"
# Using your verified password from earlier
driver = GraphDatabase.driver(uri, auth=("neo4j", "20062010")) 

def load_graph():
    with driver.session() as session:
        # Pulling the 100,000 ratings we verified earlier
        result = session.run("MATCH (u:User)-[r:RATED]->(m:Movie) RETURN u.id, m.id, r.rating")
        
        edge_index = []
        edge_attr = []
        for record in result:
            u_id = int(record["u.id"]) - 1
            m_id = int(record["m.id"]) - 1
            edge_index.append([u_id, m_id])
            edge_attr.append([float(record["r.rating"])])
            
        return (torch.tensor(edge_index, dtype=torch.long).t().contiguous(), 
                torch.tensor(edge_attr, dtype=torch.float))

# Load the data into memory
edge_index, edge_labels = load_graph()
print(f"✅ Success! Graph Loaded with {edge_index.shape[1]} connections.")

C:\Users\druth\AppData\Roaming\Python\Python312\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


✅ Success! Graph Loaded with 100000 connections.


In [3]:
# 1. Define the Model
class MovieGNN(torch.nn.Module):
    def __init__(self, num_users, num_movies):
        super().__init__()
        self.user_emb = torch.nn.Embedding(num_users, 16)
        self.movie_emb = torch.nn.Embedding(num_movies, 16)
        self.conv1 = SAGEConv(16, 16)
        self.conv2 = SAGEConv(16, 1)

    def forward(self, edge_index):
        x = torch.cat([self.user_emb.weight, self.movie_emb.weight], dim=0)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

# 2. Initialize Model and Optimizer
model = MovieGNN(num_users=943, num_movies=1682)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 3. Training Loop
print("🚀 Starting Training...")
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()
    out = model(edge_index)
    
    # We only care about the ratings that actually exist in our edge_index
    loss = F.mse_loss(out[edge_index[0]], edge_labels)
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')
print("✨ Training Complete! The model is now smart.")

🚀 Starting Training...
Epoch: 010, Loss: 5.5878
Epoch: 020, Loss: 2.2682
Epoch: 030, Loss: 1.4163
Epoch: 040, Loss: 1.4027
Epoch: 050, Loss: 1.2822
Epoch: 060, Loss: 1.2065
Epoch: 070, Loss: 1.1750
Epoch: 080, Loss: 1.1521
Epoch: 090, Loss: 1.1325
Epoch: 100, Loss: 1.1174
✨ Training Complete! The model is now smart.


In [4]:
def get_prediction(user_id, movie_id):
    model.eval()
    with torch.no_grad():
        out = model(edge_index)
        # Pull the specific user's learned preference
        prediction = out[user_id - 1].item()
        
        # Scaling the raw neural network output to a 1.0 - 5.0 scale
        final_score = max(1, min(5, prediction * 5)) 
        
        print(f"🎬 Movie Recommendation Engine")
        print(f"-------------------------------")
        print(f"Target User: {user_id}")
        print(f"Target Movie ID: {movie_id}")
        print(f"Predicted Rating: {final_score:.1f} / 5.0 stars")

# Change these numbers to any User (1-943) or Movie (1-1682)
get_prediction(user_id=85, movie_id=402)

🎬 Movie Recommendation Engine
-------------------------------
Target User: 85
Target Movie ID: 402
Predicted Rating: 5.0 / 5.0 stars


In [5]:
def get_recommendation_with_title(user_id, movie_id):
    model.eval()
    with torch.no_grad():
        out = model(edge_index)
        prediction = out[user_id - 1].item()
        final_score = max(1, min(5, prediction * 5)) 
        
        # Pull the title from Neo4j
        with driver.session() as session:
            result = session.run("MATCH (m:Movie {id: $mid}) RETURN m.title AS title", mid=movie_id)
            record = result.single()
            title = record["title"] if record else "Unknown Movie"
        
        print(f"✅ Recommendation for User {user_id}:")
        print(f"🎥 Movie: {title} (ID: {movie_id})")
        print(f"⭐ Predicted Interest: {final_score:.1f} / 5.0")

# Try it out!
get_recommendation_with_title(85, 402)

✅ Recommendation for User 85:
🎥 Movie: Ghost (1990) (ID: 402)
⭐ Predicted Interest: 5.0 / 5.0


In [6]:
# Save the model weights
torch.save(model.state_dict(), 'movie_gnn_model.pth')
print("✅ Model saved successfully as movie_gnn_model.pth")

✅ Model saved successfully as movie_gnn_model.pth
